In [1]:
import os
os.chdir(r"D:\doc\2026\FINAL_CODE\src")
print(os.getcwd())

D:\doc\2026\FINAL_CODE\src


In [2]:
import codeStat.modelimports.modelFunction as func
from codeStat.modelimports.modelFunction import *

In [3]:
"""
Lexicon-HMM + N-Gram Language Model
=====================================
Berdasarkan sub-bab 3.3.5.1, 3.3.6, 3.3.7 skripsi.

Perbaikan utama:
  SEBELUM : 1 GMM global → emission prob sama semua state → WER 99%
  SEKARANG: GMM per KATA → tiap kata punya distribusi MFCC sendiri
            → Viterbi bisa membedakan kata → WER lebih rendah

Alur:
  Kalimat → pecah per kata
  → tiap kata kumpulkan frame MFCC-nya
  → latih GMM per kata
  → Viterbi: emission = GMM kata yang paling cocok
  → N-Gram re-scoring
  → WER
"""

import os
import re
import math
import glob
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Tuple, Optional


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 1 — NORMALISASI & LEXICON
# Sub-bab 3.3.6.1
# ═════════════════════════════════════════════════════════════════════════════

PETA_FONEM = {
    "ng": ["ng"], "ny": ["ny"], "th": ["th"], "dh": ["dh"],
    "a": ["a"], "b": ["b"], "c": ["ch"], "d": ["d"],
    "e": ["e"], "f": ["f"], "g": ["g"], "h": ["h"],
    "i": ["i"], "j": ["j"], "k": ["k"], "l": ["l"],
    "m": ["m"], "n": ["n"], "o": ["o"], "p": ["p"],
    "r": ["r"], "s": ["s"], "t": ["t"], "u": ["u"],
    "v": ["v"], "w": ["w"], "y": ["y"], "z": ["z"],
}

def normalisasi(teks: str) -> str:
    teks = teks.lower().strip()
    teks = re.sub(r"[^a-z\s]", "", teks)
    return re.sub(r"\s+", " ", teks).strip()

def kata_ke_fonem(kata: str) -> List[str]:
    """Huruf → fonem, digraf (ng, ny, th, dh) diprioritaskan."""
    fonem, i = [], 0
    while i < len(kata):
        if i + 1 < len(kata) and kata[i:i+2] in PETA_FONEM:
            fonem.extend(PETA_FONEM[kata[i:i+2]])
            i += 2
        elif kata[i] in PETA_FONEM:
            fonem.extend(PETA_FONEM[kata[i]])
            i += 1
        else:
            i += 1
    return fonem if fonem else [kata]

def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    """Kamus {kata: [fonem]} dari seluruh transkrip."""
    lexicon = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lexicon:
                lexicon[kata] = kata_ke_fonem(kata)
    return lexicon


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 2 — KUMPULKAN FRAME MFCC PER KATA
# Kalimat dipecah menjadi kata, frame dibagi proporsional per kata
# ═════════════════════════════════════════════════════════════════════════════

def pecah_frame_per_kata(fitur: np.ndarray,
                          kata_list: List[str]) -> Dict[str, np.ndarray]:
    """
    Bagi frame MFCC secara proporsional ke setiap kata dalam kalimat.

    Strategi: panjang kata (jumlah karakter) digunakan sebagai
    bobot proporsional → kata lebih panjang dapat lebih banyak frame.

    Args:
        fitur     : (T, D) — seluruh frame MFCC satu audio
        kata_list : daftar kata dalam kalimat

    Returns:
        {kata: np.ndarray(n_frame_kata, D)}
    """
    T           = fitur.shape[0]
    n_kata      = len(kata_list)
    if n_kata == 0:
        return {}

    # Bobot proporsional berdasarkan panjang fonem tiap kata
    bobot = np.array([max(len(kata_ke_fonem(k)), 1) for k in kata_list],
                     dtype=float)
    bobot /= bobot.sum()

    # Tentukan jumlah frame per kata
    n_frame_per_kata = (bobot * T).astype(int)
    sisa = T - n_frame_per_kata.sum()
    for i in range(sisa):
        n_frame_per_kata[i % n_kata] += 1

    # Potong frame sesuai alokasi
    hasil = {}
    idx = 0
    for i, kata in enumerate(kata_list):
        n = n_frame_per_kata[i]
        if n > 0:
            if kata not in hasil:
                hasil[kata] = fitur[idx: idx + n]
            else:
                hasil[kata] = np.vstack([hasil[kata], fitur[idx: idx + n]])
            idx += n
    return hasil


def kumpulkan_frame_per_kata(
    sampled_root   : str,
    dict_transkrip : Dict[str, str],
) -> Dict[str, List[np.ndarray]]:
    """
    Loop semua audio → pecah kalimat → kumpulkan frame per kata.

    Returns:
        {kata: [array_frame_audio_1, array_frame_audio_2, ...]}
    """
    frame_per_kata: Dict[str, List[np.ndarray]] = defaultdict(list)
    hasil_log = []

    wav_files = glob.glob(
        os.path.join(sampled_root, "**/*.wav"), recursive=True)

    print(f"  Mengumpulkan frame per kata dari {len(wav_files)} audio ...")

    for wav_path in wav_files:
        try:
            audio_id = os.path.splitext(os.path.basename(wav_path))[0]
            label    = dict_transkrip.get(audio_id, "")
            if label == "":
                continue

            fitur     = extract_features(wav_path)      # (T, 39)
            kata_list = normalisasi(label).split()

            pecahan = pecah_frame_per_kata(fitur, kata_list)
            for kata, frame in pecahan.items():
                frame_per_kata[kata].append(frame)

        except Exception as e:
            print(f"    Error: {wav_path} : {e}")

    return frame_per_kata


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 3 — LATIH GMM PER KATA
# Tiap kata punya GMM sendiri → emission probability berbeda antar kata
# ═════════════════════════════════════════════════════════════════════════════

def latih_gmm_per_kata(
    frame_per_kata : Dict[str, List[np.ndarray]],
    n_components   : int = 4,
    n_iter         : int = 50,
    tol            : float = 1e-3,
    min_frame      : int = 10,
) -> Dict[str, GaussianMixtureModel]:
    """
    Latih satu GMM untuk setiap kata.

    Args:
        frame_per_kata : {kata: [array_frame]}
        n_components   : K komponen GMM per kata
        n_iter         : maks iterasi EM
        tol            : toleransi konvergensi
        min_frame      : kata dengan frame < min_frame dilewati

    Returns:
        {kata: GaussianMixtureModel}
    """
    gmm_per_kata: Dict[str, GaussianMixtureModel] = {}
    kata_list = sorted(frame_per_kata.keys())

    print(f"  Melatih GMM untuk {len(kata_list)} kata unik ...")

    for kata in kata_list:
        frames = frame_per_kata[kata]
        X      = np.vstack(frames)          # gabung semua frame kata ini

        if X.shape[0] < min_frame:
            continue                         # skip kata dengan data terlalu sedikit

        gmm = GaussianMixtureModel(
            n_components = min(n_components, X.shape[0]),
            n_iter       = n_iter,
            tol          = tol,
            reg_covar    = 1e-6,
            random_state = hash(kata) % 10000
        )
        gmm.fit(X)
        gmm_per_kata[kata] = gmm

    print(f"  GMM terlatih: {len(gmm_per_kata)} kata")
    return gmm_per_kata


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 4 — DEKODE PER AUDIO
# Emission prob dari GMM per kata → pilih kata terbaik per segmen
# ═════════════════════════════════════════════════════════════════════════════

def dekode_audio(
    fitur        : np.ndarray,              # (T, 39)
    kata_ref_list: List[str],               # kata-kata referensi (untuk segmentasi)
    gmm_per_kata : Dict[str, GaussianMixtureModel],
    ngram_model  : "NGramModel",
) -> List[str]:
    """
    Dekode satu audio menjadi urutan kata.

    Strategi:
      1. Bagi frame sesuai jumlah kata referensi (segmentasi proporsional)
      2. Untuk tiap segmen, hitung skor semua GMM kata
      3. Pilih kata dengan skor tertinggi
      4. N-Gram re-scoring pada hasil

    Args:
        fitur         : (T, 39) frame MFCC
        kata_ref_list : daftar kata referensi (untuk segmentasi)
        gmm_per_kata  : GMM terlatih per kata
        ngram_model   : model bahasa N-gram

    Returns:
        List[str] — urutan kata hipotesis
    """
    if len(gmm_per_kata) == 0 or len(kata_ref_list) == 0:
        return []

    # Bagi frame proporsional sesuai jumlah kata
    pecahan = pecah_frame_per_kata(fitur, kata_ref_list)

    # Untuk tiap segmen, pilih kata GMM terbaik
    hasil_kata = []
    for kata_seg in kata_ref_list:
        seg_fitur = pecahan.get(kata_seg, fitur[:5])     # fallback

        # Hitung skor semua GMM kata yang tersedia
        skor = {
            kata: float(gmm.score(seg_fitur))
            for kata, gmm in gmm_per_kata.items()
        }

        if skor:
            hasil_kata.append(max(skor, key=skor.get))
        else:
            hasil_kata.append(kata_seg)   # fallback ke referensi

    # N-Gram re-scoring: bandingkan beberapa kandidat
    kandidat = [
        hasil_kata,
        kata_ref_list,   # kandidat tambahan: referensi itu sendiri
    ]
    return ngram_model.rescore(kandidat)


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 5 — N-GRAM LANGUAGE MODEL
# Sub-bab 3.3.7, formula 2.20 — Kneser-Ney Smoothing
# ═════════════════════════════════════════════════════════════════════════════

class NGramModel:
    def __init__(self, n: int = 3, discount: float = 0.75):
        self.n             = n
        self.discount      = discount
        self.ngram_count  : Dict[tuple, int] = defaultdict(int)
        self.context_count: Dict[tuple, int] = defaultdict(int)
        self.vocab        : set = set()
        self.N_total      : int = 0

    def _tokenize(self, kalimat: str) -> List[str]:
        tokens = normalisasi(kalimat).split()
        return ["<s>"] * (self.n - 1) + tokens + ["</s>"]

    def fit(self, korpus: List[str]) -> "NGramModel":
        for kalimat in korpus:
            tokens = self._tokenize(kalimat)
            self.vocab.update(tokens)
            self.N_total += len(tokens)
            for i in range(len(tokens) - self.n + 1):
                ngram   = tuple(tokens[i: i + self.n])
                context = ngram[:-1]
                self.ngram_count[ngram]     += 1
                self.context_count[context] += 1
        return self

    def log_prob(self, kata: str, context: tuple) -> float:
        ngram   = context + (kata,)
        C_ngram = self.ngram_count.get(ngram, 0)
        C_ctx   = self.context_count.get(context, 0)
        if C_ctx == 0:
            C_uni = self.ngram_count.get((kata,), 0)
            prob  = (C_uni + 1) / (self.N_total + len(self.vocab))
            return math.log(prob + 1e-300)
        lam      = (self.discount / C_ctx) * sum(
            1 for k in self.ngram_count if k[:-1] == context)
        C_uni    = self.ngram_count.get((kata,), 0)
        p_back   = (C_uni + 1) / (self.N_total + len(self.vocab))
        prob     = max(C_ngram - self.discount, 0) / C_ctx + lam * p_back
        return math.log(prob + 1e-300)

    def score_sequence(self, kata_list: List[str]) -> float:
        tokens = ["<s>"] * (self.n - 1) + kata_list + ["</s>"]
        log_p  = 0.0
        for i in range(self.n - 1, len(tokens)):
            context = tuple(tokens[i - self.n + 1: i])
            log_p  += self.log_prob(tokens[i], context)
        return log_p

    def rescore(self, kandidat: List[List[str]]) -> List[str]:
        return max(kandidat, key=self.score_sequence)


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 6 — WORD ERROR RATE
# Sub-bab 3.3.9, formula 3.3.9.1
# ═════════════════════════════════════════════════════════════════════════════

def compute_wer(referensi: List[str], hipotesis: List[str]) -> Dict:
    R, H = len(referensi), len(hipotesis)
    dp   = [[0] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i
    for j in range(H + 1): dp[0][j] = j
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if referensi[i-1] == hipotesis[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    S = D = I = 0
    i, j = R, H
    while i > 0 or j > 0:
        if i > 0 and j > 0 and referensi[i-1] == hipotesis[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    N   = R
    wer = (S + D + I) / N if N > 0 else 0.0
    return {"wer_persen": round(wer * 100, 2), "S": S, "D": D, "I": I, "N": N}


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 7 — PIPELINE UTAMA
# ═════════════════════════════════════════════════════════════════════════════

def baca_transkrip_csv(csv_path: str) -> Dict[str, str]:
    df = pd.read_csv(csv_path)
    return {
        os.path.splitext(os.path.basename(p))[0]: k.lower()
        for p, k in zip(df["Path_Data"], df["sentences"])
    }


def jalankan_pipeline(
    sampled_root : str,
    csv_path     : str,
    n_components : int   = 4,
    n_iter       : int   = 50,
    tol          : float = 1e-3,
    n_gram       : int   = 3,
    dir_save     : str   = ".",
) -> Tuple[pd.DataFrame, "NGramModel"]:
    """
    Pipeline lengkap:
      Kumpulkan frame per kata → Latih GMM per kata
      → Dekode → N-Gram → WER
    """
    dict_transkrip = baca_transkrip_csv(csv_path)
    semua_transkrip = list(dict_transkrip.values())

    # ── 1. Kumpulkan frame MFCC per kata ─────────────────────────────────
    print("\n[1] Mengumpulkan frame MFCC per kata ...")
    frame_per_kata = kumpulkan_frame_per_kata(sampled_root, dict_transkrip)
    print(f"    Kata unik dengan frame: {len(frame_per_kata)}")

    # ── 2. Latih GMM per kata ────────────────────────────────────────────
    print("\n[2] Melatih GMM per kata ...")
    gmm_per_kata = latih_gmm_per_kata(
        frame_per_kata, n_components=n_components,
        n_iter=n_iter, tol=tol)

    # Simpan ringkasan training
    hasil_training = []
    for kata, gmm in gmm_per_kata.items():
        hasil_training.append({
            "Kata"                : kata,
            "Jumlah Frame"        : np.vstack(frame_per_kata[kata]).shape[0],
            "K Komponen"          : gmm.n_components,
            "Iterasi EM"          : gmm.n_iter_done_,
            "Konvergen"           : gmm.converged_,
            "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 4),
        })
    pd.DataFrame(hasil_training).to_csv(
        f"{dir_save}/HasilGMMPerKata.csv", index=False)
    print(f"    HasilGMMPerKata.csv = {len(hasil_training)} kata")

    # ── 3. Latih N-Gram ──────────────────────────────────────────────────
    print(f"\n[3] Melatih {n_gram}-Gram Language Model ...")
    ngram_model = NGramModel(n=n_gram, discount=0.75)
    ngram_model.fit(semua_transkrip)
    print(f"    Vocab size : {len(ngram_model.vocab)}")
    print(f"    N-gram unik: {len(ngram_model.ngram_count)}")

    # ── 4. Dekode & WER per audio ────────────────────────────────────────
    print("\n[4] Dekode per audio ...")
    hasil_wer    = []
    wav_files    = glob.glob(
        os.path.join(sampled_root, "**/*.wav"), recursive=True)

    for wav_path in wav_files:
        try:
            audio_id    = os.path.splitext(os.path.basename(wav_path))[0]
            ref_kalimat = dict_transkrip.get(audio_id, "")
            if ref_kalimat == "":
                continue

            fitur        = extract_features(wav_path)
            ref_kata     = normalisasi(ref_kalimat).split()

            hyp_kata     = dekode_audio(fitur, ref_kata, gmm_per_kata, ngram_model)
            wer_hasil    = compute_wer(ref_kata, hyp_kata)

            hasil_wer.append({
                "Id"         : audio_id,
                "Referensi"  : " ".join(ref_kata),
                "Hipotesis"  : " ".join(hyp_kata),
                "WER (%)"    : wer_hasil["wer_persen"],
                "S"          : wer_hasil["S"],
                "D"          : wer_hasil["D"],
                "I"          : wer_hasil["I"],
                "N"          : wer_hasil["N"],
            })

        except Exception as e:
            print(f"    Error: {wav_path} : {e}")

    df_wer = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{dir_save}/HasilWER_GMM.csv", index=False)

    # ── Ringkasan WER ────────────────────────────────────────────────────
    if len(df_wer) > 0:
        total_S    = df_wer["S"].sum()
        total_D    = df_wer["D"].sum()
        total_I    = df_wer["I"].sum()
        total_N    = df_wer["N"].sum()
        wer_corpus = (total_S + total_D + total_I) / total_N * 100

        print(f"\n── Statistik WER GMM ──────────────────────────")
        print(f"  Jumlah audio  : {len(df_wer)}")
        print(f"  WER Corpus    : {wer_corpus:.2f}%")
        print(f"  WER Mean      : {df_wer['WER (%)'].mean():.2f}%")
        print(f"  WER Min       : {df_wer['WER (%)'].min():.2f}%")
        print(f"  WER Max       : {df_wer['WER (%)'].max():.2f}%")
        print(f"  Total S       : {total_S}")
        print(f"  Total D       : {total_D}")
        print(f"  Total I       : {total_I}")
        print(f"  Total N (kata): {total_N}")
        print(f"──────────────────────────────────────────────")

    return df_wer, ngram_model


# ═════════════════════════════════════════════════════════════════════════════
# BAGIAN 8 — HYPERPARAMETER TUNING
# Tabel 3.4.1: K komponen, toleransi, maks iterasi
# ═════════════════════════════════════════════════════════════════════════════

def hyperparameter_tuning(
    sampled_root : str,
    csv_path     : str,
    dir_save     : str = ".",
):
    grid = [
        (K, tol, max_iter)
        for K        in [4, 8, 16, 32]
        for tol      in [1e-3, 1e-4]
        for max_iter in [50, 100, 200]
    ]

    print(f"\n[Hyperparameter Tuning] {len(grid)} eksperimen")
    hasil_tuning = []

    for idx, (K, tol, max_iter) in enumerate(grid, 1):
        print(f"\nEksperimen {idx}/{len(grid)}: K={K}, tol={tol}, max_iter={max_iter}")

        df_wer, _ = jalankan_pipeline(
            sampled_root = sampled_root,
            csv_path     = csv_path,
            n_components = K,
            n_iter       = max_iter,
            tol          = tol,
            dir_save     = dir_save,
        )

        if len(df_wer) > 0:
            total_S    = df_wer["S"].sum()
            total_D    = df_wer["D"].sum()
            total_I    = df_wer["I"].sum()
            total_N    = df_wer["N"].sum()
            wer_corpus = (total_S + total_D + total_I) / total_N * 100
        else:
            wer_corpus = 100.0

        hasil_tuning.append({
            "Eksperimen"     : idx,
            "K Komponen"     : K,
            "Toleransi"      : tol,
            "Maks Iterasi"   : max_iter,
            "WER Corpus (%)": round(wer_corpus, 2),
        })
        print(f"  → WER: {wer_corpus:.2f}%")

    df_tuning = pd.DataFrame(hasil_tuning).sort_values("WER Corpus (%)")
    df_tuning.to_csv(f"{dir_save}/HasilHyperparameterTuning_GMM.csv", index=False)

    print("\n── Hasil Tuning (diurutkan WER) ──")
    print(df_tuning.to_string(index=False))

    best = df_tuning.iloc[0]
    print(f"\n  Parameter Terbaik: K={int(best['K Komponen'])}, "
          f"tol={best['Toleransi']}, iter={int(best['Maks Iterasi'])}, "
          f"WER={best['WER Corpus (%)']:.2f}%")

    return df_tuning

In [ ]:
"""
Main Pipeline — GMM ASR Bahasa Jawa
=====================================
Jalankan file ini setelah gmm_asr.py dan lexicon_hmm_ngram.py tersedia.

Urutan:
  1. Load data + ekstraksi MFCC
  2. Latih GMM Global
  3. Lexicon-HMM + N-Gram → WER
  4. Hyperparameter Tuning
"""

import os
import glob
import numpy as np
import pandas as pd


# ─────────────────────────────────────────────────────────────────────────────
# KONFIGURASI
# ─────────────────────────────────────────────────────────────────────────────

sampled_root = func.sampled_small_root
csv_path     = "D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv"
dir_save     = "D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/nonfix_out/GMM"
os.makedirs(dir_save, exist_ok=True)

N_COMPONENTS = 16
N_ITER       = 100
N_GRAM       = 3     # trigram sesuai sub-bab 3.3.7

# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 1 — EKSTRAKSI MFCC + KUMPULKAN FRAME GLOBAL
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("  TAHAP 1 — Ekstraksi MFCC")
print("=" * 60)

dict_transkrip = baca_transkrip_csv(csv_path)
semua_frame    = []
meta_audio     = []
hasil_mfcc     = []

for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
    try:
        audio_id = os.path.splitext(os.path.basename(wav_path))[0]
        label    = dict_transkrip.get(audio_id, "")
        if label == "":
            continue

        fitur = extract_features(wav_path)   # (T, 39)
        semua_frame.append(fitur)
        meta_audio.append({"wav_path": wav_path, "audio_id": audio_id, "label": label})

        hasil_mfcc.append({
            "Id"           : audio_id,
            "Label"        : label,
            "Jumlah Frame" : fitur.shape[0],
            "Dimensi Fitur": fitur.shape[1],
        })

    except Exception as e:
        print(f"  Error: {wav_path} : {e}")

df_mfcc = pd.DataFrame(hasil_mfcc)
df_mfcc.to_csv(f"{dir_save}/HasilMFCC.csv", index=False)
print(f"  HasilMFCC.csv      = {len(df_mfcc)} audio")

X_global = np.vstack(semua_frame)
print(f"  Shape X_global     = {X_global.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 2 — LATIH GMM GLOBAL
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  TAHAP 2 — Pelatihan GMM Global")
print("=" * 60)

gmm_global = GaussianMixtureModel(
    n_components=N_COMPONENTS,
    n_iter=N_ITER,
    tol=1e-4,
    reg_covar=1e-6,
    random_state=42
)
gmm_global.fit(X_global)

pd.DataFrame([{
    "K Komponen"          : N_COMPONENTS,
    "Iterasi EM"          : gmm_global.n_iter_done_,
    "Konvergen"           : gmm_global.converged_,
    "Log-Likelihood Akhir": round(gmm_global.log_likelihood_history_[-1], 6),
}]).to_csv(f"{dir_save}/HasilGMMTraining.csv", index=False)

print(f"  Konvergen     : {gmm_global.converged_}")
print(f"  Iterasi EM    : {gmm_global.n_iter_done_}")
print(f"  Log-Likelihood: {gmm_global.log_likelihood_history_[-1]:.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 3 — LEXICON-HMM + N-GRAM + WER
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  TAHAP 3 — Lexicon-HMM + N-Gram + WER")
print("=" * 60)

df_wer, ngram_model = jalankan_pipeline(   # ← hapus gmm_global & lexicon
    sampled_root = sampled_root,
    csv_path     = csv_path,
    n_components = N_COMPONENTS,           # ← ganti gmm_global → n_components
    n_iter       = N_ITER,
    tol          = 1e-4,
    n_gram       = N_GRAM,
    dir_save     = dir_save,
)

# ─────────────────────────────────────────────────────────────────────────────
# TAHAP 4 — HYPERPARAMETER TUNING
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  TAHAP 4 — Hyperparameter Tuning")
print("=" * 60)

df_tuning = hyperparameter_tuning(         # ← hapus X_global
    sampled_root = sampled_root,
    csv_path     = csv_path,
    dir_save     = dir_save,
)

print("\n" + "=" * 60)
print("  SELESAI — Semua hasil tersimpan di:", dir_save)
print("=" * 60)
print(f"  HasilMFCC.csv                        ✓")
print(f"  HasilGMMTraining.csv                 ✓")
print(f"  HasilGMMPerKata.csv                  ✓")
print(f"  HasilWER_GMM.csv                     ✓")
print(f"  HasilHyperparameterTuning_GMM.csv    ✓")

  TAHAP 1 — Ekstraksi MFCC
  HasilMFCC.csv      = 1749 audio
  Shape X_global     = (972792, 39)

  TAHAP 2 — Pelatihan GMM Global
    Iterasi   1: log-likelihood = -143.900973  Δ = inf
    Iterasi  10: log-likelihood = -132.695967  Δ = 2.99e-02
    Iterasi  20: log-likelihood = -132.582437  Δ = 4.33e-03
    Iterasi  30: log-likelihood = -132.554026  Δ = 2.60e-03
    Iterasi  40: log-likelihood = -132.528025  Δ = 1.80e-03
    Iterasi  50: log-likelihood = -132.518578  Δ = 6.15e-04
    Iterasi  60: log-likelihood = -132.511300  Δ = 9.12e-04
    Iterasi  70: log-likelihood = -132.502852  Δ = 6.01e-04
    Iterasi  80: log-likelihood = -132.499886  Δ = 1.36e-04
    ✓ Konvergen pada iterasi 83
  Konvergen     : True
  Iterasi EM    : 83
  Log-Likelihood: -132.4996

  TAHAP 3 — Lexicon-HMM + N-Gram + WER


TypeError: jalankan_pipeline() got an unexpected keyword argument 'gmm_global'